# Exploratory Data Analysis

Adapted from https://github.com/dataprofessor/code/blob/master/python/CDD_ML_Part_2_Exploratory_Data_Analysis.ipynb

# Installing rdkit

- RDKit is an open-source cheminformatics toolkit widely used in drug discovery for manipulating and analyzing chemical structures in Python.
- It is used to compute key molecular descriptors from chemical structures.
- These descriptors help generate features that link a compound’s chemical structure to its predicted bioactivity.  
<p>This makes RDKit a central tool in machine learning–driven drug discovery and computational chemistry research.<br><br>
    
The next set of commands must be run in the terminal. DO NOT RUN IT IN THE JUPYTER NOTEBOOK

In [3]:
# This command will create a conda environment and install rdkit. Avoids 'dependency hell'
# conda create -c conda-forge -n my-rdkit-env rdkit

# Activate the conda environment
# conda activate my-rdkit-env

# Install 'ipykernel' to run jupyter-notebook that can use the packages installed in the environment
# conda install ipykernel

# Add the environment to Jupyter kernels
# python -m ipykernel install --user --name=my-rdkit-env --display-name "Python (rdkit)"

Open jupyter notebook and while creating a new notebook, choose the kernel "Python (rdkit)"

# Loading Libraries

In [5]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

# Loading the pre-processed bioactivity data

In [6]:
df = pd.read_csv("../data/bioactivity_preprocessed_data.csv")

In [7]:
df.head()

,molecule_chembl_id,canonical_smiles,standard_value,bioactivity_class
0,CHEMBL1771409,Cc1cc(N/N=C/c2ccc(O)c(O)c2)nc2ccccc12,13200.0,inactive
1,CHEMBL1771411,Cc1cc(N/N=C/c2ccncc2)nc2ccccc12,4100.0,intermediate
2,CHEMBL1933288,C[C@@H]1CCNC(=O)c2cc3ccc(C(=O)Nc4nc5ccccc5n4CC...,16.0,active
3,CHEMBL2012582,COc1cc(C(=O)N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc2c...,13.0,active
4,CHEMBL509032,COc1cc(N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc(Cl)c(N...,7.8,active


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6553 entries, 0 to 6552
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecule_chembl_id  6553 non-null   object 
 1   canonical_smiles    6551 non-null   object 
 2   standard_value      6553 non-null   float64
 3   bioactivity_class   6553 non-null   object 
dtypes: float64(1), object(3)
memory usage: 204.9+ KB


In [11]:
# Removing records/rows with missing cannonical_smiles (since they are important for the subsequent steps)
df = df[df['canonical_smiles'].notna()]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6551 entries, 0 to 6552
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecule_chembl_id  6551 non-null   object 
 1   canonical_smiles    6551 non-null   object 
 2   standard_value      6551 non-null   float64
 3   bioactivity_class   6551 non-null   object 
dtypes: float64(1), object(3)
memory usage: 255.9+ KB


# Calculate Lipinski Descriptors

Christopher Lipinski, a scientist at Pfizer, developed a simple set of guidelines known as the Rule of Five to help evaluate whether a compound is likely to become a successful oral drug. These rules focus on key physical and chemical properties that affect how the drug behaves in the body, including how well it is absorbed, distributed, metabolized, and excreted (collectively known as ADME).

Lipinski's Rule of Five states that, in general, an orally active drug should have:

- A molecular weight under 500 daltons (small size helps absorption),
- A lipophilicity (LogP) less than 5 (balance between water and fat solubility),
- No more than 5 hydrogen bond donors (which affect solubility and membrane crossing),
- No more than 10 hydrogen bond acceptors (which also influence solubility).

Following these rules increases the chances that a compound will be absorbed well when taken by mouth and reach effective levels in the bloodstream. While not a guarantee of success, these descriptors serve as a practical filter to prioritize drug-like candidates early in the discovery process, saving time and resources.

This concept helps guide the design and selection of compounds that are more likely to be effective as oral medications.

In [8]:
# Inspired by: https://codeocean.com/explore/capsules?query=tag:data-curation

def lipinski(smiles, verbose=False):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem) 
        moldata.append(mol)
       
    baseData= np.arange(1,1)
    i=0  
    for mol in moldata:        
       
        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)
           
        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors])   
    
        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1      
    
    columnNames=["MW","LogP","NumHDonors","NumHAcceptors"]   
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)
    
    return descriptors

In [12]:
df_lipinski = lipinski(df.canonical_smiles)

In [13]:
df_lipinski

,MW,LogP,NumHDonors,NumHAcceptors
0,293.326,3.40042,3.0,5.0
1,262.316,3.38422,1.0,4.0
2,458.566,3.88950,2.0,6.0
3,570.698,3.43870,1.0,9.0
4,614.216,5.02410,2.0,10.0
...,...,...,...,...
6546,407.404,2.92170,2.0,9.0
6547,421.431,3.57710,2.0,9.0
6548,365.363,3.65040,2.0,7.0
6549,379.390,4.00338,2.0,7.0


# Combining dataframes

Adding the Lipinski descriptors to the pre-processed Bioactivity data

In [14]:
df_combined = pd.concat([df,df_lipinski], axis=1)
df_combined.head()

,molecule_chembl_id,canonical_smiles,standard_value,bioactivity_class,MW,LogP,NumHDonors,NumHAcceptors
0,CHEMBL1771409,Cc1cc(N/N=C/c2ccc(O)c(O)c2)nc2ccccc12,13200.0,inactive,293.326,3.40042,3.0,5.0
1,CHEMBL1771411,Cc1cc(N/N=C/c2ccncc2)nc2ccccc12,4100.0,intermediate,262.316,3.38422,1.0,4.0
2,CHEMBL1933288,C[C@@H]1CCNC(=O)c2cc3ccc(C(=O)Nc4nc5ccccc5n4CC...,16.0,active,458.566,3.88950,2.0,6.0
3,CHEMBL2012582,COc1cc(C(=O)N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc2c...,13.0,active,570.698,3.43870,1.0,9.0
4,CHEMBL509032,COc1cc(N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc(Cl)c(N...,7.8,active,614.216,5.02410,2.0,10.0
